In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
from patchcore.memory_bank import MemoryBank
from patchcore.feature_extractor import ResNetFeatureExtractor
from patchcore.anomaly_scoring import compute_anomaly_score
from patchcore.coreset import coreset_sampling
from patchcore.feature_extractor import ResNetFeatureExtractor, extract_patches

In [3]:
import torch
from torchvision import transforms
from PIL import Image
from tqdm import tqdm

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

extractor = ResNetFeatureExtractor().to(device)
extractor.eval()

transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.ToTensor()
])

In [5]:
import os

DATA_ROOT = r"C:\Users\ADMIN\Documents\CV_Proj\Data\mvtec"

categories = os.listdir(DATA_ROOT)
print(categories)

['bottle', 'capsule', 'hazelnut', 'metal_nut']


In [6]:
import torch.nn.functional as F

def train_patchcore_for_category(category):

    print(f"\nTraining for category: {category}")

    train_path = os.path.join(DATA_ROOT, category, "train", "good")

    memory_bank = MemoryBank()

    image_files = os.listdir(train_path)

    for img_name in tqdm(image_files):

        img_path = os.path.join(train_path, img_name)

        image = Image.open(img_path).convert("RGB")
        image = transform(image).unsqueeze(0).to(device)

        with torch.no_grad():
            feat2, feat3 = extractor(image)

        # Resize feat3 to match feat2 resolution
        feat3_resized = F.interpolate(feat3, size=feat2.shape[2:], mode='bilinear', align_corners=False)
        # Now extract patches
        patches2 = extract_patches(feat2)
        patches3 = extract_patches(feat3_resized)
        # Now they will match
        patches = torch.cat([patches2, patches3], dim=1)

        memory_bank.add(patches)

    bank = memory_bank.build()

    # Apply coreset
    coreset_bank = coreset_sampling(bank, sampling_ratio=0.1)

    # Save memory bank
    save_path = f"../models/{category}_memory.pt"
    torch.save(coreset_bank, save_path)

    print(f"Saved memory bank: {save_path}")

In [7]:
for category in categories:
    train_patchcore_for_category(category)


Training for category: bottle


100%|██████████| 209/209 [00:09<00:00, 21.87it/s]


Saved memory bank: ../models/bottle_memory.pt

Training for category: capsule


100%|██████████| 219/219 [00:15<00:00, 14.15it/s]


Saved memory bank: ../models/capsule_memory.pt

Training for category: hazelnut


100%|██████████| 391/391 [00:28<00:00, 13.90it/s]


Saved memory bank: ../models/hazelnut_memory.pt

Training for category: metal_nut


100%|██████████| 220/220 [00:10<00:00, 20.90it/s]


Saved memory bank: ../models/metal_nut_memory.pt


In [8]:
def load_memory_bank(category):

    path = f"../models/{category}_memory.pt"

    bank = torch.load(path, map_location=device)

    return bank

In [9]:
def run_inference(category, image_path):

    bank = load_memory_bank(category)

    image = Image.open(image_path).convert("RGB")
    img_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        feat2, feat3 = extractor(img_tensor)

    patches2 = extract_patches(feat2)
    patches3 = extract_patches(feat3)

    patches = torch.cat([patches2, patches3], dim=1)

    scores = compute_anomaly_score(patches, bank)

    return scores, img_tensor

In [10]:
print("patches shape:", patches.shape)
print("bank shape:", bank.shape)

NameError: name 'patches' is not defined